In [2]:
from langchain_groq import ChatGroq
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import yfinance as yf
from langchain.agents import AgentType, initialize_agent
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool
from langchain_groq import ChatGroq

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

from pymongo.operations import SearchIndexModel
from langchain_community.vectorstores import AtlasDB
# from langchain_mongodb import MongoDBAtlasVectorSearch
# from pymongo import MongoClient
from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser
import PyPDF2
import pandas as pd
from langchain_core.documents import Document
from typing import Iterable
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter
)
from langchain.text_splitter import NLTKTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables.utils import ConfigurableFieldSpec
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [32]:
"""
Stock Market Analysis Agent using LangGraph and Groq API

This code creates a langgraph agent that:
1. Takes user queries about stocks
2. Fetches real-time stock data from Alpha Vantage API
3. Analyzes the data using Groq's LLM API
4. Returns insights to the user

"""

import os
import json
import requests
from datetime import datetime, timedelta
from typing import Dict, List, Any, Annotated, TypedDict, Sequence
from pydantic import BaseModel, Field

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

import langgraph
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolExecutor
from langgraph.prebuilt.tool_node import ToolNode

# Define API keys (in production, use environment variables)
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "your-groq-api-key")
ALPHAVANTAGE_API_KEY = os.environ.get("ALPHAVANTAGE_API_KEY", "your-alphavantage-api-key")

# Define State schema
class AgentState(TypedDict):
    """Represents the state of our agent."""
    messages: List[Any]
    tools_output: List[Dict[str, Any]]
    next: str

# Initialize the Groq Model
llm = ChatGroq(
    model_name="llama3-70b-8192",  # Using a capable model for financial analysis
    api_key=GROQ_API_KEY,
    temperature=0.1  # Lower temperature for more deterministic responses
)

# Define Stock Analysis Tools



In [38]:
llm.invoke("WHat is MSFT?")

AIMessage(content="MSFT is the stock ticker symbol for Microsoft Corporation. It is listed on the NASDAQ stock exchange under the ticker symbol MSFT.\n\nMicrosoft is a multinational technology company that develops, manufactures, licenses, and supports a wide range of software products, services, and devices. The company was founded in 1975 by Bill Gates and Paul Allen.\n\nMicrosoft is one of the largest and most successful technology companies in the world, with a diverse product portfolio that includes:\n\n* Operating systems (Windows)\n* Productivity software (Microsoft Office)\n* Server software (Windows Server)\n* Database software (Microsoft Azure)\n* Artificial intelligence and machine learning (Xbox, Azure)\n* Cloud computing (Microsoft Azure)\n* Gaming consoles (Xbox)\n* Online advertising (Bing)\n\nMicrosoft is considered one of the Big Five technology companies in the United States, along with Amazon, Apple, Alphabet (Google), and Facebook.\n\nAs a publicly traded company, M

In [ ]:
"""
Simple Stock Market Analysis Agent using LangGraph and Groq API

This code creates a minimal langgraph agent that:
1. Takes user queries about stocks
2. Extracts the stock ticker from the query
3. Uses Groq's LLM API to provide stock information and analysis
4. Maintains state with ticker and date information
"""

import os
from datetime import datetime
from typing import Dict, List, Any, TypedDict

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, END

# Define API key (in production, use environment variables)
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "your-groq-api-key")

# Define simple State schema
class StockState(TypedDict):
    """Represents the state of our stock analysis agent."""
    messages: List[Any]
    stock_ticker: str  # Tracks current stock ticker
    date: str  # Tracks current date for analysis

# Initialize the Groq Model
llm = ChatGroq(
    model_name="llama3-70b-8192",  # Using a capable model for financial analysis
    api_key=GROQ_API_KEY,
    temperature=0.2  # Lower temperature for more consistent responses
)

# Define system prompt for stock analysis
system_prompt = """You are a helpful stock market analysis assistant.

Your task is to provide information and insights about stocks based on the user's query.
You should respond with general information about the stock, including:
- Basic company information
- Recent stock price trends (in general terms)
- Market sentiment and outlook
- Key business factors affecting the stock
- Notable recent events (if any)

Since you don't have access to real-time stock data, provide general information based on your training data.
Always make it clear that your information is not current and should not be used for investment decisions.

Today's date is {current_date}.
The stock ticker currently being discussed is: {stock_ticker}
"""

# Create the chat prompt template
stock_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("placeholder", "{chat_history}"),
    ("human", "{user_input}")
])

# Define the single LLM node function
def process_query(state: StockState) -> Dict[str, Any]:
    """Process the user query and generate a response using the LLM."""
    messages = state["messages"]
    current_date = state.get("date", datetime.now().strftime("%Y-%m-%d"))
    stock_ticker = state.get("stock_ticker", "Unknown")
    
    # Extract the most recent user message
    user_message = messages[-1].content if messages and isinstance(messages[-1], HumanMessage) else ""
    
    # Extract previous messages for chat history
    chat_history = messages[:-1] if len(messages) > 1 else []
    
    # Try to extract stock ticker from the message if not already set
    if stock_ticker == "Unknown" or not stock_ticker:
        # Simple extraction logic - look for capital letters that might be tickers
        words = user_message.split()
        for word in words:
            # Check if word is in all caps and between 1-5 characters - likely a ticker
            if word.isupper() and 1 <= len(word) <= 5 and word.isalpha():
                stock_ticker = word
                break
    
    # Format the LLM prompt with current state
    formatted_prompt = stock_prompt.format(
        current_date=current_date,
        stock_ticker=stock_ticker,
        chat_history=chat_history,
        user_input=user_message
    )
    
    # Get response from LLM
    response = llm.invoke(formatted_prompt)
    
    # Update the state with the response and extracted ticker
    updated_messages = messages + [AIMessage(content=response.content)]
    
    return {
        "messages": updated_messages,
        "stock_ticker": stock_ticker,
        "date": current_date
    }

# Build the graph with a single node
workflow = StateGraph(StockState)
workflow.add_node("process", process_query)
workflow.set_entry_point("process")
workflow.add_edge("process", END)

# Compile the graph
stock_agent = workflow.compile()

# Function to run the agent
def run_stock_agent(query: str, stock_ticker: str = "", date: str = "") -> Dict[str, Any]:
    """Run the stock analysis agent with a user query."""
    if not date:
        date = datetime.now().strftime("%Y-%m-%d")
        
    messages = [HumanMessage(content=query)]
    result = stock_agent.invoke({
        "messages": messages,
        "stock_ticker": stock_ticker,
        "date": date
    })
    
    return result

# Example usage
if __name__ == "__main__":
    # Example queries to test the agent
    example_queries = [
        "Tell me about AAPL stock",
        "What's going on with TSLA?",
        "How is Microsoft doing in the market?",
        "Give me some information about AMZN"
    ]
    
    for query in example_queries:
        print(f"\n--- Query: {query} ---")
        result = run_stock_agent(query)
        print(f"Stock Ticker: {result['stock_ticker']}")
        print(f"Response: {result['messages'][-1].content}")

In [40]:
class MessageState(TypedDict):
    messages: List[Any]
    stock_ticker: str
    date: str

In [44]:
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "your-groq-api-key")
llm = ChatGroq(
    model_name="llama3-70b-8192",  # Using a capable model for financial analysis
    api_key=GROQ_API_KEY,
    temperature=0.2,  # Lower temperature for more consistent responses,
    verbose=True ,
    max_retries=2
)

In [45]:
llm.invoke("What is MSFT?")

AIMessage(content="MSFT is the stock ticker symbol for Microsoft Corporation. It is listed on the NASDAQ stock exchange.\n\nMicrosoft is a multinational technology company that develops, manufactures, licenses, and supports a wide range of software products, services, and devices. The company's best-known software products include the Windows operating system, Office productivity suite, and Outlook email client. Microsoft is also a leading provider of cloud computing services, artificial intelligence, and gaming consoles (Xbox).\n\nAs one of the largest and most influential technology companies in the world, MSFT is a widely followed stock and a component of the Dow Jones Industrial Average (DJIA) and the S&P 500 index.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 134, 'prompt_tokens': 15, 'total_tokens': 149, 'completion_time': 0.409503041, 'prompt_time': 0.000380035, 'queue_time': 0.055082055, 'total_time': 0.409883076}, 'model_name': 'llama3-70b-81

In [58]:
from groq import Groq

# Replace with your actual Groq API key
client = Groq(api_key=GROQ_API_KEY)

# Step 1: Define the custom tool
tools = [
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiplies two numbers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"}
                },
                "required": ["a", "b"]
            }
        }
    }
]

# Step 2: Send initial message to Groq with tool binding
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",  # Tool-callable model
    messages=[{"role": "user", "content": "what is 67/19?"}],
    tools=tools,
    tool_choice="auto"
)
print(response)

# Step 3: Process tool call
tool_calls = response.choices[0].message.tool_calls
print(response.choices[0].message)
messages = [{"role": "user", "content": "What is 6 times 7?"}]




ChatCompletion(id='chatcmpl-b474900d-2e5b-424b-b767-0d0e7b422e64', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='<function=multiply({"a": 67, "b": 1/19})</function>', role='assistant', function_call=None, reasoning=None, tool_calls=None))], created=1747735480, model='llama-3.3-70b-versatile', object='chat.completion', system_fingerprint='fp_2ddfbb0da0', usage=CompletionUsage(completion_tokens=22, prompt_tokens=231, total_tokens=253, completion_time=0.08, prompt_time=0.014197197, queue_time=0.060343573000000005, total_time=0.094197197), usage_breakdown={'models': None}, x_groq={'id': 'req_01jvpje2gtevy9a2hh5pjt3xma'})
ChatCompletionMessage(content='<function=multiply({"a": 67, "b": 1/19})</function>', role='assistant', function_call=None, reasoning=None, tool_calls=None)


In [60]:
if tool_calls:
    print(tool_calls)
    for tool_call in tool_calls:
        if tool_call.function.name == "multiply":
            args = tool_call.function.arguments
            print(args)
            print(type(args))
            args = json.loads(tool_call.function.arguments)  # Parse JSON string to dict
            a = args.get("a", 1)
            b = args.get("b", 1)
            result = a * b
            tool_response_message = {
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": "multiply",
                "content": str(result)
            }
            messages.append(tool_response_message)

    # Step 4: Send follow-up message with tool result
    final_response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages
    )

    print("Final model response:")
    print(final_response.choices[0].message.content)

else:
    print("Model did not call any tool.")
    print("Response:", response.choices[0].message.content)

Model did not call any tool.
Response: <function=multiply({"a": 67, "b": 1/19})</function>
